In [17]:
# Define the mapping again
education_years_map = {
    "Baccalauréat": 12,
    "High School Diploma": 12,
    "Niveau Terminal": 12,
    "Terminal Level": 12,
    "Licence (LMD), Bac + 3": 15,
    "Bachelor's Degree (LMD), Bac + 3": 15,
    "Bachelor's Degree": 15,
    "Master 1, Licence  Bac + 4": 16,
    "Master 1, Bachelor's Degree  Bac + 4": 16,
    "Master 2, Ingéniorat, Bac + 5": 17,
    "Master 2, Engineering, Bac + 5": 17,
    "Ingéniorat": 17,
    "Doctorat": 20,
    "PhD": 20,
    "Magistère Bac + 7": 21,
    "Magisterium Bac + 7": 21,
    "TS Bac +2": 14,
    "Technicien Supérieur": 14,
    "Formation Professionnelle": 13,           
    "Universitaire Sans Diplôme": 13,          
    "Non Diplômante": 11,                      
    "Non Degree": 11,                          
    "Certification": 12,                       
    "Niveau Secondaire": 10,                   
    "University Without A Degree": 13,         
    "No Formal Education": 0,                 
}

# Define the extractor
def extract_max_education_years(education_string):
    if pd.isna(education_string):
        return np.nan
    levels = [level.strip() for level in str(education_string).replace(",", "|").split("|")]
    max_years = np.nan
    for level in levels:
        found = False
        for key in education_years_map.keys():
            if key.lower() in level.lower():
                val = education_years_map[key]
                if not np.isnan(val):
                    max_years = max(max_years, val) if not np.isnan(max_years) else val
                found = True
                break
        if not found:
            import re
            match = re.search(r'Bac\s*\+?\s*(\d+)', level)
            if match:
                years = 12 + int(match.group(1))
                max_years = max(max_years, years) if not np.isnan(max_years) else years
    return max_years

# Apply the transformation
df['education_years'] = df['education requirement'].apply(extract_max_education_years)

# Move 'description' column to the end if it exists
if 'description' in df.columns:
    cols = [col for col in df.columns if col != 'description'] + ['description']
    df = df[cols]

# Save to new CSV
output_path = 'emploitic_job_listings_with_education_years.csv'
df.to_csv(output_path, index=False)

output_path


'emploitic_job_listings_with_education_years.csv'

In [18]:
import pandas as pd

# Load the dataset
df = pd.read_csv("data_with_education_years.csv")

# Define a function to infer job type from job title or description
def infer_job_type(row):
    text = f"{row.get('job title', '')} {row.get('description', '')}".lower()
    if 'temps partiel' in text or 'part-time' in text or 'temps part time' in text:
        return 'Part-time'
    elif 'temps plein' in text or 'full-time' in text or 'temps plein' in text:
        return 'Full-time'
    else:
        # Fallback: guess based on common patterns
        if 'freelance' in text or 'indépendant' in text:
            return 'Part-time'
        return 'Full-time'  # Default assumption

# Apply function to create the job_type column
df['job_type'] = df.apply(infer_job_type, axis=1)

# Save updated file
df.to_csv("data_with_job_type.csv", index=False)

print("✅ job_type column added and file saved as data_with_job_type.csv")


✅ job_type column added and file saved as data_with_job_type.csv
